In [1]:
import pandas as pd
import numpy as nm

In [2]:
paitents = pd.read_csv('patients.csv')
labs = pd.read_csv('labs.csv')
outcomes = pd.read_csv('outcomes.csv')
diagnoses = pd.read_csv('diagnoses.csv')

In [3]:
paitents

,PatientID,Name,Age,Gender,DiagnosisID,AdmissionDate,DischargeDate,OutcomeID,TreatmentCost
0,1,Edward Montes,68,M,2,2025-03-22,2025-04-04,1,4443
1,2,Jessica Zimmerman,81,M,2,2025-03-31,2025-04-02,2,4265
2,3,Gina Alexander,58,M,6,2025-01-08,2025-03-13,1,4188
3,4,Lori Green,44,F,3,2025-04-03,2025-04-05,1,9783
4,5,Jack Logan,72,F,9,2025-06-03,2025-06-04,1,5005
...,...,...,...,...,...,...,...,...,...
195,196,Shelly Dominguez,72,M,10,2025-04-28,2025-05-02,3,3311
196,197,Destiny Johnson,58,M,2,2025-03-06,2025-07-08,3,5191
197,198,Catherine Case,65,M,3,2025-05-26,2025-06-04,2,8957
198,199,Dennis Wolfe,42,F,2,2025-06-26,2025-07-05,2,4538


In [4]:
paitents = paitents.merge(diagnoses, on = 'DiagnosisID')
paitents = paitents.merge(outcomes, on = 'OutcomeID')

In [5]:
paitents['AdmissionDate'] = pd.to_datetime(paitents['AdmissionDate'])
paitents['DischargeDate'] = pd.to_datetime(paitents['DischargeDate'])
paitents['LengthOfStay'] = (paitents['DischargeDate'] - paitents['AdmissionDate']).dt.days

In [6]:
paitents

,PatientID,Name,Age,Gender,DiagnosisID,AdmissionDate,DischargeDate,OutcomeID,TreatmentCost,DiagnosisName,OutcomeName,LengthOfStay
0,1,Edward Montes,68,M,2,2025-03-22,2025-04-04,1,4443,Diabetes,Recovered,13
1,2,Jessica Zimmerman,81,M,2,2025-03-31,2025-04-02,2,4265,Diabetes,Complicated,2
2,3,Gina Alexander,58,M,6,2025-01-08,2025-03-13,1,4188,COPD,Recovered,64
3,4,Lori Green,44,F,3,2025-04-03,2025-04-05,1,9783,Heart Disease,Recovered,2
4,5,Jack Logan,72,F,9,2025-06-03,2025-06-04,1,5005,Kidney Disease,Recovered,1
...,...,...,...,...,...,...,...,...,...,...,...,...
195,196,Shelly Dominguez,72,M,10,2025-04-28,2025-05-02,3,3311,Liver Disease,Deceased,4
196,197,Destiny Johnson,58,M,2,2025-03-06,2025-07-08,3,5191,Diabetes,Deceased,124
197,198,Catherine Case,65,M,3,2025-05-26,2025-06-04,2,8957,Heart Disease,Complicated,9
198,199,Dennis Wolfe,42,F,2,2025-06-26,2025-07-05,2,4538,Diabetes,Complicated,9


In [20]:
paitents['OutcomeEncoded'] = paitents['OutcomeName'].map({'Recovered': 0, 'Complicated': 1, 'Deceased':1})

In [21]:
paitents['HighRisk'] = nm.where((paitents['Age']>65) | (paitents['OutcomeName'].isin(['Complicated','Deceased'])), 1, 0)

In [22]:
abnormal_conditions = {
    'Blood Sugar': lambda x : x > 120,
    'Cholesterol': lambda x : x > 200,
    'Hemoglobin' : lambda x : x < 13
}

def count_abnormal_labs(paitent_id):
    paitents_labs = labs[labs['PatientID'] == paitent_id]
    count = 0
    for test_name, condition in abnormal_conditions.items():
        test_results = paitents_labs[paitents_labs['TestName'] == test_name]
        count += test_results['Result'].apply(condition).sum()
    return count
paitents['AbnormalLabCount'] = paitents['PatientID'].apply(count_abnormal_labs)

Model Training

In [23]:
features = paitents[['Age','LengthOfStay','TreatmentCost']]
target = paitents['OutcomeEncoded']

In [24]:
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(features, target, test_size = 0.3, random_state = 42)

In [25]:
from sklearn.linear_model import LogisticRegression
model = LogisticRegression(max_iter = 500)
model.fit(x_train, y_train)

LogisticRegression(max_iter=500)

In [26]:
from sklearn.metrics import classification_report, confusion_matrix
y_pred = model.predict(x_test)
print("Classification Report: ")
print(classification_report(y_test, y_pred))

Classification Report: 
              precision    recall  f1-score   support

           0       0.00      0.00      0.00        17
           1       0.72      1.00      0.83        43

    accuracy                           0.72        60
   macro avg       0.36      0.50      0.42        60
weighted avg       0.51      0.72      0.60        60



c:\ProgramData\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\ProgramData\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\ProgramData\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [27]:
import joblib
joblib.dump(model, 'Risk_Model.ipynb')

['Risk_Model.ipynb']